 # Section 1: Setup, Unified Paths & Global Settings

 This cell loads essential utilities, configures global Pandas display parameters,

 silences silent-downcasting warnings, and dynamically maps project directories.

In [1]:
# 3. Path Selection Toggle

# USE_LOCAL_DATA = False  # Set False for GLOBAL_DATA_DIR
USE_LOCAL_DATA = True  # Set False for GLOBAL_DATA_DIR

In [2]:
%load_ext autoreload
%autoreload 2

import ast
from pathlib import Path

import numpy as np
import pandas as pd

from core.contracts import EngineInput, FilterPack
from core.paths import (
    GLOBAL_DATA_DIR,
    GLOBAL_PROCESSED_DIR,
    LOCAL_DATA_DIR,
    NOTEBOOKS_RLVR_ROOT,
    OUTPUT_DIR,
    PROJECT_ROOT,
)
from core.settings import TradingConfig
from core.utils import SystemUtils as SU
from data_pipeline import generate_features
from walk_forward import (
    AlphaEngine,
    create_walk_forward_analyzer,
    run_headless_simulation,
)

# 1. Global Configuration Options
config = TradingConfig()

# 2. Display Settings & Silence Warnings
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 3000)

# 3. Path Selection Toggle
# USE_LOCAL_DATA = False  # Set False for GLOBAL_DATA_DIR

data_dir = LOCAL_DATA_DIR if USE_LOCAL_DATA else GLOBAL_DATA_DIR
processed_dir = GLOBAL_PROCESSED_DIR
output_dir = OUTPUT_DIR

# 4. Define Target File Paths
ohlcv_filename = (
    "df_ohlcv.parquet" if USE_LOCAL_DATA else "df_OHLCV_stocks_etfs.parquet"
)
fed_filename = (
    "df_fed.parquet"
    if USE_LOCAL_DATA
    else "High_Yield_Spread_T10Y2Y_Spread.parquet"
)

raw_paths = {
    "df_ohlcv": data_dir / ohlcv_filename,
    "df_fed": data_dir / fed_filename,
    "df_indices": data_dir / "df_indices.parquet",
}

features_df_path = data_dir / "features_df.parquet"
macro_df_path = data_dir / "macro_df.parquet"

print(
    f"Paths Resolved:\n"
    f" - Active Mode:          {'LOCAL' if USE_LOCAL_DATA else 'GLOBAL'}\n"
    f" - Active Data Dir:       {data_dir}\n"
    f" - Global Processed Dir: {GLOBAL_PROCESSED_DIR}\n"
    f" - Output Dir:           {OUTPUT_DIR}\n"
)

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output

Paths Resolved:
 - Active Mode:          LOCAL
 - Active Data Dir:       C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
 - Global Processed Dir: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
 - Output Dir:           C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output



# Section 2: Ingest Datasets & Generate Features

Loads raw input files and retrieves pre-calculated features or runs feature generation.

In [3]:
# Load Raw Files
raw_dfs = {}

for name, path in raw_paths.items():
    if path.exists():
        raw_dfs[name] = pd.read_parquet(path)
        print(f"[OK] Successfully loaded {name:<10} from {path.name}")
    else:
        print(f"[ERROR] File not found: {path}")

df_ohlcv = raw_dfs.get("df_ohlcv")
df_fed = raw_dfs.get("df_fed")
df_indices = raw_dfs.get("df_indices")

[OK] Successfully loaded df_ohlcv   from df_ohlcv.parquet
[OK] Successfully loaded df_fed     from df_fed.parquet
[OK] Successfully loaded df_indices from df_indices.parquet


In [4]:
# Load Processed Features or Generate If Missing / Running Global
if USE_LOCAL_DATA and features_df_path.exists() and macro_df_path.exists():
    features_df = pd.read_parquet(features_df_path)
    macro_df = pd.read_parquet(macro_df_path)
    print(f"[OK] Loaded features_df from local disk | Shape: {features_df.shape}")
    print(f"[OK] Loaded macro_df from local disk    | Shape: {macro_df.shape}")
else:
    reason = (
        "USE_LOCAL_DATA is False"
        if not USE_LOCAL_DATA
        else "Local feature files missing"
    )
    print(f"[INFO] {reason}. Generating features (~6 mins)...")

    features_df, macro_df = generate_features(
        df_ohlcv=df_ohlcv, df_indices=df_indices, df_fed=df_fed, config=config
    )
    print(f"[OK] Generated features_df  | Shape: {features_df.shape}")
    print(f"[OK] Generated macro_df     | Shape: {macro_df.shape}")

[OK] Loaded features_df from local disk | Shape: (7864460, 22)
[OK] Loaded macro_df from local disk    | Shape: (7434, 11)


In [5]:
# Cache Processed DataFrames to GLOBAL_PROCESSED_DIR when running Global
if not USE_LOCAL_DATA:
    GLOBAL_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Saving DataFrames to: {GLOBAL_PROCESSED_DIR}\n")

    dfs_to_save = {
        "df_ohlcv.parquet": df_ohlcv,
        "df_indices.parquet": df_indices,
        "df_fed.parquet": df_fed,
        "features_df.parquet": features_df,
        "macro_df.parquet": macro_df,
    }

    for filename, df in dfs_to_save.items():
        if df is not None:
            save_path = GLOBAL_PROCESSED_DIR / filename
            df.to_parquet(save_path)
            print(f"[OK] Saved {filename:<20} | Shape: {df.shape}")
        else:
            print(f"[SKIP] {filename:<19} | DataFrame missing in memory")

In [6]:
###############################
###############################

In [7]:
print("🚀 Unstacking Wide Matrices...")

# 1. Unstack directly.
# (Since df_ohlcv is already a perfect grid of master_dates x tickers,
# unstacking automatically creates perfectly aligned wide dataframes).
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0)
df_trp_wide = features_df["TRP"].unstack(level=0)

print("Applying terminal NaN replacements for model inputs...")

# 2. Terminal Fills ONLY.
# Note: 0-to-NaN conversion and bounded ffills were ALREADY applied upstream to df_ohlcv.
# Applying ffill here would incorrectly double your max_data_gap_ffill limit!
df_close_wide = df_close_wide.fillna(config.nan_price_replacement)
df_atrp_wide = df_atrp_wide.fillna(0.0)
df_trp_wide = df_trp_wide.fillna(0.0)

print("All wide dataframes unstacked and prepared for downstream modeling.")
print(
    f"[OK] Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

🚀 Unstacking Wide Matrices...
Applying terminal NaN replacements for model inputs...
All wide dataframes unstacked and prepared for downstream modeling.
[OK] Pipeline Complete. Tickers: 1573, Days: 7434


 # Section 3: In-Memory Structural Previews

 Outputs head and tail previews of the core dataframes.

In [ ]:
print("\n" + "=" * 50)
print("DATAFRAME STRUCTURAL PREVIEWS (HEAD & TAIL)")
print("=" * 50)

# 1. Preview wide matrix prices
if "df_close_wide" in locals():
    print("\n=== df_close_wide (Prices) ===")
    print("--- HEAD (First 3 Rows) ---")
    display(df_close_wide.head(3))
    print("\n--- TAIL (Last 3 Rows) ---")
    display(df_close_wide.tail(3))

# 2. Preview multi-index features
if "features_df" in locals():
    print("\n=== features_df (Multi-Index Features) ===")
    print("--- HEAD (First 3 Rows) ---")
    display(features_df.head(3))
    print("\n--- TAIL (Last 3 Rows) ---")
    display(features_df.tail(3))

# 3. Preview macro data
if "macro_df" in locals():
    print("\n=== macro_df (Macro Data) ===")
    print("--- HEAD (First 3 Rows) ---")
    display(macro_df.head(3))
    print("\n--- TAIL (Last 3 Rows) ---")
    display(macro_df.tail(3))

 # Section 4: Engine Initialization & Stage 1 Analysis

In [9]:
# Instantiate Master Engine
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("Master AlphaEngine Ready.")

Master AlphaEngine Ready.


In [10]:
import pandas as pd

target_date = pd.Timestamp("2025-09-01")

# Use level=1 for the date level (since level names aren't 'timestamp')
dates_in_index = df_ohlcv.index.get_level_values(1).normalize().unique()
print(f"2025-09-01 in index: {target_date in dates_in_index}")

# Check if date exists (one-liner)
any_match = (df_ohlcv.index.get_level_values(1).normalize() == target_date).any()
print(f"Any match: {any_match}")

# If you want to see which tickers have data on that date:
if any_match:
    mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
    tickers_on_date = df_ohlcv.index.get_level_values(0)[mask].unique()
    print(f"Tickers on 2025-09-01: {tickers_on_date}")
    print(f"Count: {len(tickers_on_date)}")

# Extract all data for that specific date
if any_match:
    # Using boolean mask (works regardless of index sorting)
    mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
    df_target = df_ohlcv[mask]
    print(df_target.head())

2025-09-01 in index: False
Any match: False


In [ ]:
# Configuration for Interactive UI (Stage 1 Setup)
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=189,
    holding_period=5,
    metric="Sharpe (TRP)",
    benchmark_ticker=config.benchmark_ticker,
    rank_start=1,
    rank_end=200,
    debug=False,
)

analyzer1, stage1_pack = create_walk_forward_analyzer(
    engine, _inputs, universe_subset=None
)
analyzer1.show()

In [ ]:
# Execute Headless Agent Simulation
print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df.style.format("{:.6f}"))
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

 # Section 5: Stage 2 - Cascade Filter Execution

In [ ]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# LAUNCH STAGE 2 (Cascade on the Stage 1 subset)
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
# Parse and map results from Analyzer 2
result = SU.map_analyzer(analyzer2)

In [ ]:
# Test different methods to extract resulting target tickers
_tickers = SU.fetch(4, result)
print(f"_tickers (by index): {_tickers}\n")

_tickers = SU.fetch("tickers", result)
print(f"_tickers (by key): {_tickers}\n")

print(f'result[4]["obj"]: {result[4]["obj"]}')

 # Section 6: Verification, Exports & Auditing

In [ ]:
# Export analytical trace logs to Excel
SU.export_audit_to_excel(
    audit_pack=analyzer2.last_run, filename="Audit_Verification_Report.xlsx"
)

In [ ]:
# Export last-run stacked OHLCV ticker records
SU.export_last_run_tickers_data_to_csv(
    analyzer=analyzer2,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename="last_run_tickers_stacked.csv",
)

 # Section 7: Slicing Tests & Dictionary Parsing

In [ ]:
# Verify captured ticker names
_tickers

In [ ]:
# Test various pandas multi-index slicing configurations
test_tickers = ["NVDA", "GOOG"]
idx = pd.IndexSlice
columns = ["Adj Close", "Volume"]

# Slice 1: All dates, all columns
display(df_ohlcv.loc[idx[test_tickers]].copy().head(3))

# Slice 2: specific start date, all columns
display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], :].head(3))

# Slice 3: specific start date, selected columns
display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], columns].head(3))

# Slice 4: specific start/end range, all columns
display(
    df_ohlcv.loc[
        idx[test_tickers, pd.Timestamp("2026-04-01") : pd.Timestamp("2026-04-10")], :
    ].head(3)
)

In [ ]:
# Test custom combined dictionary extraction
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=_tickers,
    date_start="2026-05-01",
    date_end=None,
    verbose=False,
)

 # Section 8: Finviz Ingestion & Filtering

In [ ]:
# Load latest updated Finviz dataset dynamically
def load_latest_finviz_parquet(data_dir: Path) -> pd.DataFrame:
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"
    files = list(data_dir.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {data_dir.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Load the file using the unified data_dir path
df_finviz = load_latest_finviz_parquet(GLOBAL_DATA_DIR)
print(
    f"df_finviz raw records shape: {df_finviz.shape if df_finviz is not None else None}"
)

In [ ]:
display(df_finviz.head())
display(df_finviz.tail())

In [ ]:
raw_text = """
BP, USO, SHEL, KEYS, NOK, INTC, NBIS, CIEN, APA, MTZ,
SNDK, JAZZ, PR, UTHR, EQIX, VLO, SU, CVE, COHR, CLS,
"""

# Split by comma, strip whitespace, and filter out any empty strings
ticker_list = [item.strip() for item in raw_text.split(",") if item.strip()]

print(ticker_list)

In [ ]:
# Set test target tickers
target_tickers = ["GLW", "NOK", "DELL", "NVDA", "GOOG"]

In [ ]:
# Clean index extraction using ast exception parsing if tickers are missing
try:
    result = df_finviz.loc[target_tickers]
except KeyError as e:
    error_msg = str(e)
    missing_str = error_msg.replace(" not in index", "").strip('"').strip("'")
    missing_tickers = ast.literal_eval(missing_str)

    print(f"Missing tickers: {missing_tickers}")

    # Strip out the missing symbols and retry slicing
    target_tickers_in_finviz = [t for t in target_tickers if t not in missing_tickers]
    result = df_finviz.loc[target_tickers_in_finviz].copy()
    result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

print(f"result shape: {result.shape}")

In [ ]:
# Display filtered Finviz information and schema details
result.info()

In [ ]:
# Check specific target column (Market Capitalization)
result["MktCap AUM, M"]

In [ ]:
# Sort records descendingly by Market Capitalization
result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

In [ ]:
# View final sorted DataFrame
print(result)

In [ ]:
# # Export processed dataframe output directly to trash file
# result.to_csv(output_dir / "_trash.csv")